<a href="https://colab.research.google.com/github/Dhairyasummi/voiceagent/blob/main/text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

INSTALL NECESSARY LIBRARIES


In [ ]:
try:
  import datasets, evaluate, accelerate
  import gradio as gr
except ModuleNotFoundError:
  !pip install -U datasets evaluate accelerate gradio
  import datasets, evaluate, accelerate
  import gradio as gr

import random

import numpy as np
import pandas as pd

import torch
import transformers

print(f"Using transformer version: {transformers.__version__}")
print(f"Using torch  version: {torch.__version__}")
print(f"Usingg dataset version: {datasets.__version__}")

Using transformer version: 5.0.0
Using torch  version: 2.10.0+cu128
Usingg dataset version: 4.8.5


GETTING A DATASET

building food and not food text classification model

In [ ]:
from datasets import load_dataset

dataset = load_dataset(path="mrdbourke/learn_hf_food_not_food_image_captions")
dataset

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/250 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 250
    })
})

In [ ]:
# what features are there
dataset.column_names

{'train': ['text', 'label']}

In [ ]:
# accessing the training split
dataset["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 250
})

In [ ]:
dataset["train"][0]

{'text': 'Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
 'label': 'food'}

### inspect random samples


In [ ]:
import random

random_indexs = random.sample(range(len(dataset["train"])), 5)
print(random_indexs)

random_samples = dataset["train"][random_indexs]

print(f"[INFO] Random samples from dataset:\n")
for text, label in zip(random_samples["text"], random_samples["label"]):
  print(f"Text: {text} | Label: {label}")

[197, 214, 249, 201, 130]
[INFO] Random samples from dataset:

Text: Pizza with a stuffed crust, oozing with cheese | Label: food
Text: Microwave oven ready for use on a kitchen counter | Label: not_food
Text: Taking a nap on a hammock, a man has his dog snuggled up next to him | Label: not_food
Text: Dishwasher installed in a kitchen | Label: not_food
Text: Comforting lamb curry bowl, featuring tender lamb slow-cooked in a flavorful sauce with cumin and coriander, garnished with toasted cumin seeds. | Label: food


In [ ]:
# get unique label values
dataset["train"].unique("label")

['food', 'not_food']

In [ ]:
# check the count of each label
from collections import Counter

Counter(dataset["train"]["label"])

Counter({'food': 125, 'not_food': 125})

In [ ]:
# turning our dataset into dataframe and get a random sample
food_not_food_df = pd.DataFrame(dataset["train"])
food_not_food_df.sample(7)

,text,label
183,Washing machine and dryer side by side in a la...,not_food
225,"Spicy chickpea curry bowl, featuring nutty chi...",food
22,Nigiri sushi topped with fresh salmon and tuna...,food
105,Set of measuring spoons hung on a rack,not_food
79,"Creamy spinach and potato curry, featuring flu...",food
75,"Sushi with a spicy kick, featuring jalapeno pe...",food
248,"Family gathered around a dining table, laughin...",not_food


In [ ]:
food_not_food_df["label"].value_counts()

,count
label,
food,125
not_food,125


### Preparing data for text classification

we want to :
 1. Tokenize out text - turn our text into numbers
 2. Create a train/test split- want to train our model on the training split and want to evaluate our model on the test split

In [ ]:
dataset["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 250
})

In [ ]:
# create a mapping for labels to the numeric value
id2label = {0 : "not_food", 1 : "food"}
label2id = {"not_food" : 0, "food" : 1}

print(id2label)
print(label2id)

{0: 'not_food', 1: 'food'}
{'not_food': 0, 'food': 1}


In [ ]:
# create the programmatically automatically from dataset
id2label = {idx: label for idx, label in enumerate(dataset["train"].unique("label"))}
label2id = {label: idx for idx, label in enumerate(dataset["train"].unique("label"))}

print(id2label)
print(label2id)


{0: 'food', 1: 'not_food'}
{'food': 0, 'not_food': 1}


{0: 'food', 1: 'not_food'}